# 03 – แผนที่พืชที่เหมาะสมในประเทศไทย (XGBoost (โมเดลใหม่))
ใช้โมเดล XGBoost (โมเดลใหม่) จาก `enriched_crop_pipeline.pkl` ทำนายพืชที่เหมาะกับแต่ละจุดในไทย
โดยเชื่อม `climate_normal.xml` (ภูมิอากาศ) กับ `soil.xml` (ดิน) ด้วยพิกัดที่ใกล้ที่สุด แล้วปักหมุดบนแผนที่ด้วย **folium**

ไฟล์ที่ต้องอยู่โฟลเดอร์เดียวกับ notebook นี้
- จาก repo: `enriched_crop_pipeline.pkl`, `Crop_recommendation.csv`, `climate_normal.xml`, `soil.xml`, `enriched_crop_dataset_cleaned.csv`
- ไฟล์เสริม: `thailand_provinces.geojson` (ขอบเขตจังหวัด), `thailand_actual_crop_area.csv` (พื้นที่ปลูกจริงจาก สศก.)

In [22]:
# ติดตั้งแพ็กเกจ (รันครั้งแรกครั้งเดียว) — ใช้ได้ทั้ง Jupyter ปกติ, Colab และ Jupyter ในเบราว์เซอร์ (Pyodide)
import sys
PKGS = ['folium', 'pandas', 'scikit-learn', 'joblib', 'requests', 'openpyxl', 'kagglehub', 'xgboost']
if 'pyodide' in sys.modules:                       # Jupyter ในเบราว์เซอร์ (JupyterLite)
    import piplite
    for p in PKGS:
        try:
            await piplite.install(p)
        except Exception:
            print('ติดตั้งไม่ได้ (ข้ามได้):', p)
else:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS])
print('ติดตั้งเสร็จ')

ติดตั้งเสร็จ


## 1. โหลดชุดข้อมูล Kaggle (หลัก) และโมเดล

In [23]:
import json, glob, os, warnings
import numpy as np, pandas as pd, joblib
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')

try:
    import kagglehub
    path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")
    kag = pd.read_csv(glob.glob(path + '/*.csv')[0])
    print("โหลดจาก Kaggle:", path)
except Exception:
    kag = pd.read_csv('Crop_recommendation.csv')      # สำรอง: ไฟล์เดียวกันใน repo
    print("ใช้ไฟล์ใน repo แทน")

PIPELINE = 'enriched_crop_pipeline.pkl'
OUT = '_xgboost'                                    # ต่อท้ายชื่อไฟล์ผลลัพธ์ เพื่อไม่ให้ทับกันระหว่างโมเดล
pipe = joblib.load(PIPELINE)
model, scaler, le, MODEL_FEATS = pipe['model'], pipe['scaler'], pipe['label_encoder'], list(pipe['feature_names'])
FEATS = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']    # 7 ปัจจัยหลัก ใช้แสดงผลและเทียบช่วงกับ Kaggle
LOC_FEATS = [f for f in MODEL_FEATS if f.startswith('location_')]        # คอลัมน์ one-hot ชื่อสถานี (มีเฉพาะโมเดลใหม่)
CROPS = list(le.classes_)
print(type(model).__name__, '|', len(MODEL_FEATS), 'features |', len(LOC_FEATS), 'คอลัมน์ location |', len(CROPS), 'พืช')

โหลดจาก Kaggle: C:\Users\User\.cache\kagglehub\datasets\atharvaingle\crop-recommendation-dataset\versions\1
XGBClassifier | 59 features | 49 คอลัมน์ location | 22 พืช


## 2. อ่านข้อมูลภูมิอากาศ (สถานีกรมอุตุฯ)

In [24]:
rows = []
for s in ET.parse('climate_normal.xml').getroot().iter('StationClimateNormal'):
    nv = s.find('NormalValue'); el = nv.find('Element')
    vals = [pd.to_numeric(c.text, errors='coerce') for c in nv if c.tag.startswith('ValueOn')]
    rows.append(dict(station=s.findtext('StationName'),
                     lat=pd.to_numeric(s.findtext('Latitude'), errors='coerce'),
                     lon=pd.to_numeric(s.findtext('Longitude'), errors='coerce'),
                     element=el.text, vals=vals))
clim = pd.DataFrame(rows).dropna(subset=['lat'])

def wide(name):   # ตาราง สถานี x 12 เดือน
    d = clim[clim.element == name].drop_duplicates('station').set_index('station')
    return pd.DataFrame(d.vals.tolist(), index=d.index)
T, H, R = wide('Mean Dry-bulb Temperature'), wide('Mean Relative Humidity'), wide('Total Rainfall')
st = clim.drop_duplicates('station').set_index('station')[['lat', 'lon']]
st = st.loc[st.index.intersection(T.dropna().index).intersection(H.dropna().index).intersection(R.dropna().index)]
print('สถานีที่ข้อมูลครบ:', len(st)); st.head()

สถานีที่ข้อมูลครบ: 120


,lat,lon
station,,
MAE HONG SON,19.299,97.976
MAE SARIANG,18.167,97.933
CHIANG RAI,19.961,99.881
CHAING RAI AGROMET.,19.871,99.783
PHAYAO,19.133,99.900


## 3. อ่านข้อมูลดิน
ดึงจาก API ของกรมพัฒนาที่ดินก่อน (`SOIL_URL`) โดยลองแบ่งหน้า/กรองจังหวัดเพื่อให้ได้มากกว่า 1,000 รายการ แล้วเก็บไว้ที่ `soil_all.json`
ถ้าต่อ API ไม่ได้จะใช้ไฟล์ `soil.xml` (เนื้อในเป็น JSON) · ไม่มีค่า N จึงประมาณ `N = OM% × 20` (แก้ได้ที่ `N_PER_OM`)

In [25]:
try:
    import requests
except ImportError:
    requests = None                    # ไม่มี requests → ใช้ soil.xml แทนการดึงจาก API
SOIL_URL = "https://osd101.ldd.go.th/api/data.php"
REFRESH = False            # True = ดึงจาก API ใหม่ แม้มี soil_all.json อยู่แล้ว
PROVINCES = ['กระบี่', 'กรุงเทพมหานคร', 'กาญจนบุรี', 'กาฬสินธุ์', 'กำแพงเพชร', 'ขอนแก่น', 'จันทบุรี', 'ฉะเชิงเทรา', 'ชลบุรี', 'ชัยนาท', 'ชัยภูมิ', 'ชุมพร', 'ตรัง', 'ตราด', 'ตาก', 'นครนายก', 'นครปฐม', 'นครพนม', 'นครราชสีมา', 'นครศรีธรรมราช', 'นครสวรรค์', 'นนทบุรี', 'นราธิวาส', 'น่าน', 'บึงกาฬ', 'บุรีรัมย์', 'ปทุมธานี', 'ประจวบคีรีขันธ์', 'ปราจีนบุรี', 'ปัตตานี', 'พระนครศรีอยุธยา', 'พะเยา', 'พังงา', 'พัทลุง', 'พิจิตร', 'พิษณุโลก', 'ภูเก็ต', 'มหาสารคาม', 'มุกดาหาร', 'ยะลา', 'ยโสธร', 'ระนอง', 'ระยอง', 'ราชบุรี', 'ร้อยเอ็ด', 'ลพบุรี', 'ลำปาง', 'ลำพูน', 'ศรีสะเกษ', 'สกลนคร', 'สงขลา', 'สตูล', 'สมุทรปราการ', 'สมุทรสงคราม', 'สมุทรสาคร', 'สระบุรี', 'สระแก้ว', 'สิงห์บุรี', 'สุพรรณบุรี', 'สุราษฎร์ธานี', 'สุรินทร์', 'สุโขทัย', 'หนองคาย', 'หนองบัวลำภู', 'อำนาจเจริญ', 'อุดรธานี', 'อุตรดิตถ์', 'อุทัยธานี', 'อุบลราชธานี', 'อ่างทอง', 'เชียงราย', 'เชียงใหม่', 'เพชรบุรี', 'เพชรบูรณ์', 'เลย', 'แพร่', 'แม่ฮ่องสอน']

def _get(params=None):
    if requests is None: raise RuntimeError('ไม่มีแพ็กเกจ requests')
    r = requests.get(SOIL_URL, params=params, timeout=60); r.raise_for_status()
    d = r.json()
    if isinstance(d, dict):                      # เผื่อ API ห่อ list ไว้ใน dict
        d = next((v for v in d.values() if isinstance(v, list)), [])
    return [x for x in d if isinstance(x, dict) and 'parameter' in x]

def _key(x): return json.dumps(x, sort_keys=True, ensure_ascii=False)

def fetch_soil():
    base = _get(); seen = {_key(x): x for x in base}
    print(f"API ค่าเริ่มต้น: {len(base)} รายการ")
    def new_rows(rows):
        n = 0
        for x in rows:
            k = _key(x)
            if k not in seen: seen[k] = x; n += 1
        return n
    # 1) ขอทีเดียวให้มากขึ้น
    for p in ({'limit': 100000}, {'per_page': 100000}, {'pagesize': 100000}):
        try:
            n = new_rows(_get(p))
            if n: print(f"  {p} ได้เพิ่ม {n} รายการ")
        except Exception: pass
    # 2) แบ่งหน้า
    step = max(len(base), 1)
    for name, first, inc in (('page', 2, 1), ('offset', step, step), ('start', step, step)):
        v, got = first, 0
        for _ in range(500):
            try: n = new_rows(_get({name: v}))
            except Exception: break
            if not n: break
            got += n; v += inc
        if got: print(f"  แบ่งหน้าด้วย '{name}' ได้เพิ่ม {got} รายการ")
    # 3) กรองทีละจังหวัด
    have = {x.get('province') for x in seen.values()}
    probe = next((p for p in PROVINCES if p not in have), None)
    for name in ('province', 'prov', 'changwat'):
        if not probe: break
        try: rows = _get({name: probe})
        except Exception: continue
        if rows and all(x.get('province') == probe for x in rows):
            got = 0
            for p in PROVINCES:
                try: got += new_rows(_get({name: p}))
                except Exception: pass
            print(f"  กรองจังหวัดด้วย '{name}' ได้เพิ่ม {got} รายการ"); break
    return list(seen.values())

if os.path.exists('soil_all.json') and not REFRESH:
    raw = json.load(open('soil_all.json', encoding='utf-8')); print("ใช้ soil_all.json ที่เก็บไว้")
else:
    try:
        raw = fetch_soil()
        json.dump(raw, open('soil_all.json', 'w', encoding='utf-8'), ensure_ascii=False)
    except Exception as e:
        print("ต่อ API ไม่ได้ (", type(e).__name__, ") → ใช้ไฟล์ soil.xml")
        raw = json.load(open('soil.xml', encoding='utf-8'))
print("รายการดินทั้งหมด:", len(raw))

N_PER_OM = 20
s = pd.DataFrame(raw)
s['parameter'] = s.parameter.str.replace(r'<sup>.*?</sup>', '', regex=True).str.strip()
s['v'] = pd.to_numeric(s.value, errors='coerce')
s['lat'] = pd.to_numeric(s.latitude, errors='coerce'); s['lon'] = pd.to_numeric(s.longitude, errors='coerce')
s = s[s.lat.between(5, 21) & s.lon.between(97, 106)]          # ตัดพิกัดที่อยู่นอกประเทศไทย
s['p'] = s.parameter.map(lambda t: 'P' if '(P)' in t else 'K' if '(K)' in t else 'ph' if '(pH)' in t else 'OM' if '(OM)' in t else None)
keys = ['province', 'amphur', 'district', 'lat', 'lon']
s[keys[:3]] = s[keys[:3]].fillna('')
site = s.dropna(subset=['p', 'v']).pivot_table(index=keys, columns='p', values='v', aggfunc='median').reset_index()
for col in ['P', 'K', 'ph', 'OM']:      # ค่าที่ไม่ได้วัด เติมด้วยมัธยฐานจังหวัด แล้วมัธยฐานรวม
    if col not in site: site[col] = np.nan
    site[col] = site[col].fillna(site.groupby('province')[col].transform('median')).fillna(site[col].median())
site['N'] = site.OM * N_PER_OM
print('จุดดิน:', len(site), '| จังหวัด:', site.province.nunique(), 'จาก 77')
missing = sorted(set(PROVINCES) - set(site.province))
print('จังหวัดที่ยังไม่มีข้อมูลดิน:', len(missing), missing[:12], '...' if len(missing) > 12 else '')
site.head()

ใช้ soil_all.json ที่เก็บไว้
รายการดินทั้งหมด: 943
จุดดิน: 80 | จังหวัด: 23 จาก 77
จังหวัดที่ยังไม่มีข้อมูลดิน: 54 ['กาฬสินธุ์', 'ขอนแก่น', 'จันทบุรี', 'ฉะเชิงเทรา', 'ชลบุรี', 'ชัยภูมิ', 'ตราด', 'ตาก', 'นครนายก', 'นครปฐม', 'นครพนม', 'นครราชสีมา'] ...


p,province,amphur,district,lat,lon,K,OM,P,ph,N
0,กระบี่,เขาพนม,เขาดิน,8.34200,99.167000,350.0,2.135,104.56,5.215,42.7
1,กรุงเทพมหานคร,บางขุนเทียน,ท่าข้าม,13.71615,100.680761,356.0,2.620,33.00,4.200,52.4
2,กรุงเทพมหานคร,ประเวศ,ประเวศ,13.71615,100.680761,229.0,11.940,42.00,7.300,238.8
3,กรุงเทพมหานคร,หนองจอก,หนองจอก,13.71615,100.680761,223.0,2.130,6.00,3.900,42.6
4,กาญจนบุรี,ด่านมะขามเตี้ย,หนองไผ่,13.80800,99.440000,57.5,1.330,39.10,7.500,26.6


## 4. เชื่อมข้อมูล (foreign key = จุดที่ใกล้ที่สุด)
จุดดิน → สถานีอากาศใกล้สุด และ สถานีอากาศ → จุดดินใกล้สุด

In [26]:
def hav(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = map(np.radians, (la1, lo1, la2, lo2))
    a = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))
D = hav(site.lat.values[:, None], site.lon.values[:, None], st.lat.values[None], st.lon.values[None])

pts = []
for i, r in site.iterrows():
    j = D[i].argmin()
    pts.append(dict(kind='soil', name=f"ต.{r.district} อ.{r.amphur} จ.{r.province}", lat=r.lat, lon=r.lon,
                    station=st.index[j], soil_site=i, dist_km=D[i, j]))
for j, (nm, r) in enumerate(st.iterrows()):
    i = D[:, j].argmin()
    pts.append(dict(kind='station', name=nm.title(), lat=r.lat, lon=r.lon,
                    station=nm, soil_site=i, dist_km=D[i, j]))
P = pd.DataFrame(pts)
print(P.kind.value_counts().to_dict(), '| ดินไกลกว่า 100 กม.:', (P.dist_km > 100).sum(), 'จุด'); P.head()

{'station': 120, 'soil': 80} | ดินไกลกว่า 100 กม.: 38 จุด


,kind,name,lat,lon,station,soil_site,dist_km
0,soil,ต.เขาดิน อ.เขาพนม จ.กระบี่,8.34200,99.167000,PHRA SANG,0,27.256579
1,soil,ต.ท่าข้าม อ.บางขุนเทียน จ.กรุงเทพมหานคร,13.71615,100.680761,BANG NA AGROMET.,1,9.814904
2,soil,ต.ประเวศ อ.ประเวศ จ.กรุงเทพมหานคร,13.71615,100.680761,BANG NA AGROMET.,2,9.814904
3,soil,ต.หนองจอก อ.หนองจอก จ.กรุงเทพมหานคร,13.71615,100.680761,BANG NA AGROMET.,3,9.814904
4,soil,ต.หนองไผ่ อ.ด่านมะขามเตี้ย จ.กาญจนบุรี,13.80800,99.440000,KANCHANA BURI,4,26.055696


## 5. ทำนายด้วยโมเดล
เลือกช่วงเวลาได้ที่ `SEASON` · ฝนใช้ค่าเฉลี่ยรายเดือนของช่วงนั้น · ค่าที่เกินช่วง Kaggle ถูกตัดให้อยู่ในช่วง

โค้ดสร้างคอลัมน์ให้ครบตามชื่อใน `feature_names` ของโมเดล จึงใช้ได้ทั้งโมเดลเก่า (7 คอลัมน์) และโมเดลใหม่ (59 คอลัมน์)
- `N, P, K, ph` = ค่าดินของจุด · `temperature, humidity, rainfall` หรือ `*_original` = อากาศของช่วงที่เลือก
- `*_xml` = ค่าทั้งปีของสถานี (อุณหภูมิ/ความชื้นเฉลี่ย, ฝนรวมทั้งปี) คำนวณแบบเดียวกับ notebook 00
- `location_*` = 1 ที่ชื่อสถานีของจุดนั้น · สถานีที่โมเดลไม่เคยเห็นใช้ชื่อสถานีที่โมเดลรู้จักซึ่งอยู่ใกล้ที่สุด

In [27]:
SEASONS = {'rainy': [4,5,6,7,8,9], 'dry': [10,11,0,1,2,3], 'year': list(range(12))}
SEASON = 'rainy'          # 'rainy' = พ.ค.–ต.ค. | 'dry' = พ.ย.–เม.ย. | 'year' = ทั้งปี

LOC_OF = {}
if LOC_FEATS:
    # ชื่อ location ที่โมเดลรู้จัก = คอลัมน์ one-hot + ตัวที่ถูก drop_first ซึ่งหาได้จากไฟล์ train ถ้ามี
    KNOWN_LOC = [f[len('location_'):] for f in LOC_FEATS]
    if os.path.exists('enriched_crop_dataset_cleaned.csv'):
        KNOWN_LOC = sorted(pd.read_csv('enriched_crop_dataset_cleaned.csv').location.unique())
    known = [s_ for s_ in st.index if s_.upper() in KNOWN_LOC]
    Dk = hav(st.lat.values[:, None], st.lon.values[:, None], st.loc[known].lat.values[None], st.loc[known].lon.values[None])
    LOC_OF = {s_: (s_.upper() if s_.upper() in KNOWN_LOC else known[Dk[i].argmin()].upper()) for i, s_ in enumerate(st.index)}
    print(f"สถานีที่โมเดลรู้จักชื่อ: {len(known)} จาก {len(st)} · ที่เหลือใช้ชื่อสถานีใกล้สุดที่โมเดลรู้จัก")

def predict(season):
    m = SEASONS[season]; ss = site.loc[P.soil_site]
    X = pd.DataFrame({'N': ss.N.values, 'P': ss.P.values, 'K': ss.K.values,
                      'temperature': T.loc[P.station, m].mean(axis=1).values,
                      'humidity': H.loc[P.station, m].mean(axis=1).values,
                      'ph': ss.ph.values,
                      'rainfall': R.loc[P.station, m].mean(axis=1).values})[FEATS]
    Xc = X.clip(kag[FEATS].min(), kag[FEATS].max(), axis=1)
    annual = {'temperature_xml': T.loc[P.station].mean(axis=1).values, 'humidity_xml': H.loc[P.station].mean(axis=1).values,
              'rainfall_xml': R.loc[P.station].sum(axis=1).values}
    loc = P.station.map(LOC_OF).values
    M = pd.DataFrame(index=range(len(P)), columns=MODEL_FEATS, dtype=float)
    for f in MODEL_FEATS:                     # สร้างคอลัมน์ตามชื่อและลำดับที่โมเดลต้องการ
        if f in Xc: M[f] = Xc[f].values
        elif f.endswith('_original'): M[f] = Xc[f[:-len('_original')]].values
        elif f in annual: M[f] = annual[f]
        elif f.startswith('location_'): M[f] = (loc == f[len('location_'):]).astype(float)
        else: raise ValueError(f'ไม่รู้วิธีสร้าง feature: {f}')
    proba = model.predict_proba(scaler.transform(M))
    order = np.argsort(-proba, axis=1)[:, :3]
    res = pd.concat([P, X.round(2)], axis=1)
    for k in range(3):
        res[f'top{k+1}'] = [CROPS[c] for c in order[:, k]]
        res[f'prob{k+1}'] = proba[np.arange(len(P)), order[:, k]].round(3)
    return res

res = predict(SEASON)
res.to_csv(f'thailand_crop_predictions{OUT}.csv', index=False, encoding='utf-8-sig')
print(res.top1.value_counts()); res.head()

สถานีที่โมเดลรู้จักชื่อ: 50 จาก 120 · ที่เหลือใช้ชื่อสถานีใกล้สุดที่โมเดลรู้จัก
top1
rice          63
coconut       46
jute          36
grapes        21
apple          7
banana         7
pigeonpeas     6
papaya         4
watermelon     4
chickpea       3
mothbeans      2
blackgram      1
Name: count, dtype: int64


,kind,name,lat,lon,station,soil_site,dist_km,N,P,K,temperature,humidity,ph,rainfall,top1,prob1,top2,prob2,top3,prob3
0,soil,ต.เขาดิน อ.เขาพนม จ.กระบี่,8.34200,99.167000,PHRA SANG,0,27.256579,42.7,104.56,350.0,26.98,85.27,5.22,177.32,pigeonpeas,0.648,jute,0.089,chickpea,0.031
1,soil,ต.ท่าข้าม อ.บางขุนเทียน จ.กรุงเทพมหานคร,13.71615,100.680761,BANG NA AGROMET.,1,9.814904,52.4,33.00,356.0,29.14,76.89,4.20,226.03,rice,0.900,jute,0.019,pigeonpeas,0.010
2,soil,ต.ประเวศ อ.ประเวศ จ.กรุงเทพมหานคร,13.71615,100.680761,BANG NA AGROMET.,2,9.814904,238.8,42.00,229.0,29.14,76.89,7.30,226.03,rice,0.859,cotton,0.097,jute,0.016
3,soil,ต.หนองจอก อ.หนองจอก จ.กรุงเทพมหานคร,13.71615,100.680761,BANG NA AGROMET.,3,9.814904,42.6,6.00,223.0,29.14,76.89,3.90,226.03,rice,0.861,coconut,0.037,jute,0.018
4,soil,ต.หนองไผ่ อ.ด่านมะขามเตี้ย จ.กาญจนบุรี,13.80800,99.440000,KANCHANA BURI,4,26.055696,26.6,39.10,57.5,28.49,74.28,7.50,143.77,papaya,0.085,coffee,0.069,coconut,0.058


## 6. แผนที่ปักหมุด (แบบพิมพ์ชื่อพืช)
เปลี่ยน `CROP` เป็นชื่อพืชที่ต้องการ (เช่น `'rice'`, `'maize'`, `'coconut'`) หรือ `'all'` เพื่อดูพืชอันดับ 1 ของทุกจุด
ขอบเขียว = อันดับ 1 · ขอบส้ม = อันดับ 2–3 · เส้นประ = ค่าดินยืมจากจุดที่ไกลกว่า 100 กม. · แบบมีเมนูเลือกอยู่ข้อ 8

In [28]:
import folium
ICON = {'apple':'🍎','banana':'🍌','blackgram':'🫘','chickpea':'🫘','coconut':'🥥','coffee':'☕','cotton':'☁️','grapes':'🍇',
        'jute':'🌿','kidneybeans':'🫘','lentil':'🫘','maize':'🌽','mango':'🥭','mothbeans':'🫘','mungbean':'🫛','muskmelon':'🍈',
        'orange':'🍊','papaya':'🧡','pigeonpeas':'🫛','pomegranate':'🔴','rice':'🌾','watermelon':'🍉'}
TH = {'apple':'แอปเปิล','banana':'กล้วย','blackgram':'ถั่วดำ','chickpea':'ถั่วลูกไก่','coconut':'มะพร้าว','coffee':'กาแฟ','cotton':'ฝ้าย',
      'grapes':'องุ่น','jute':'ปอกระเจา','kidneybeans':'ถั่วแดง','lentil':'ถั่วเลนทิล','maize':'ข้าวโพด','mango':'มะม่วง','mothbeans':'ถั่วมอธ',
      'mungbean':'ถั่วเขียว','muskmelon':'แคนตาลูป','orange':'ส้ม','papaya':'มะละกอ','pigeonpeas':'ถั่วแระ','pomegranate':'ทับทิม',
      'rice':'ข้าว','watermelon':'แตงโม'}

PROV_URL = 'https://raw.githubusercontent.com/chingchai/OpenGISData-Thailand/master/provinces.geojson'
try:
    if os.path.exists('thailand_provinces.geojson'):
        PROV_GEO = json.load(open('thailand_provinces.geojson', encoding='utf-8'))
    else:
        PROV_GEO = requests.get(PROV_URL, timeout=60).json() if requests else None
except Exception:
    PROV_GEO = None; print('โหลดขอบเขตจังหวัดไม่ได้ จะแสดงเฉพาะแผนที่ฐาน')

def base_map():
    # พื้นหลัง: ขอบเขตจังหวัดจากไฟล์ (ไม่ต้องใช้ API key) + แผนที่ฐาน Esri ที่เปิด/ปิดได้
    m = folium.Map(location=[13.2, 101.2], zoom_start=6, tiles=None)
    folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
                     attr='Tiles &copy; Esri', name='แผนที่ฐาน (Esri)').add_to(m)
    if PROV_GEO:
        folium.GeoJson(PROV_GEO, name='ขอบเขตจังหวัด', tooltip=folium.GeoJsonTooltip(['pro_th'], labels=False),
                       style_function=lambda f: {'color': '#7d8f80', 'weight': 1, 'fillColor': '#e3eadf', 'fillOpacity': 0.6}).add_to(m)
    return m

def crop_map(crop='rice', season='rainy'):
    res = predict(season)
    m = base_map(); n = 0
    for r in res.itertuples():
        tops = [r.top1, r.top2, r.top3]
        if crop == 'all': c, rank = r.top1, 1
        elif crop in tops: c, rank = crop, tops.index(crop) + 1
        else: continue
        n += 1
        color = '#1f7a4d' if rank == 1 else '#b7791f'
        style = 'dashed' if r.dist_km > 100 else 'solid'
        html = (f'<div style="width:26px;height:26px;border-radius:50%;background:#fff;border:2px {style} {color};'
                f'text-align:center;line-height:22px;font-size:14px">{ICON[c]}</div>')
        popup = (f"<b>{r.name}</b><br>1. {TH[r.top1]} ({r.prob1:.0%})<br>2. {TH[r.top2]} ({r.prob2:.0%})<br>3. {TH[r.top3]} ({r.prob3:.0%})"
                 f"<hr style='margin:4px 0'>N {r.N} · P {r.P} · K {r.K} · pH {r.ph}<br>"
                 f"อุณหภูมิ {r.temperature}°C · ความชื้น {r.humidity}% · ฝน {r.rainfall} มม./เดือน<br>"
                 f"สถานีอากาศ: {r.station.title()} · ระยะดิน–สถานี {r.dist_km:.0f} กม.")
        folium.Marker([r.lat, r.lon], icon=folium.DivIcon(html=html, icon_size=(26, 26), icon_anchor=(13, 13)),
                      tooltip=f"{r.name} – {TH[c]} อันดับ {rank}", popup=folium.Popup(popup, max_width=320)).add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)
    print(f"{TH.get(crop, 'ทุกพืช')} · {season} · {n} จุด")
    return m

CROP = 'rice'
m = crop_map(CROP, SEASON)
m

ข้าว · rainy · 81 จุด


## 7. ตรวจโมเดลกับพื้นที่ปลูกจริง (สถิติ สศก.)
ใช้ `thailand_actual_crop_area.csv` (ข้าวนาปี 2567, ข้าวนาปรัง 2568, ข้าวโพดเลี้ยงสัตว์ 2567, กาแฟ 2568, มะพร้าว 2568)
นับว่าจังหวัด "ปลูกจริง" เมื่อเนื้อที่ถึงเกณฑ์ใน `ACTUAL` · นับว่าโมเดล "แนะนำ" เมื่อพืชนั้นติด 3 อันดับแรกอย่างน้อย 1 จุดในจังหวัด

In [29]:
# จังหวัดของแต่ละจุด: จุดดินใช้ชื่อจังหวัดในข้อมูล, สถานีอากาศหาจากขอบเขตจังหวัด
def _in_ring(x, y, ring):
    inside = False; j = len(ring) - 1
    for i in range(len(ring)):
        xi, yi = ring[i][0], ring[i][1]; xj, yj = ring[j][0], ring[j][1]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / (yj - yi) + xi: inside = not inside
        j = i
    return inside

def province_of(lat, lon):
    best, bd = None, 1e9
    for f in (PROV_GEO or {}).get('features', []):
        g = f['geometry']; polys = g['coordinates'] if g['type'] == 'MultiPolygon' else [g['coordinates']]
        for poly in polys:
            if _in_ring(lon, lat, poly[0]): return f['properties']['pro_th']
            d = min((x - lon) ** 2 + (y - lat) ** 2 for x, y, *_ in poly[0])
            if d < bd: best, bd = f['properties']['pro_th'], d
    return best            # จุดริมชายฝั่ง/ชายแดนที่ตกนอกเส้นขอบ ใช้จังหวัดที่ใกล้ที่สุด

P['province'] = [site.province[r.soil_site] if r.kind == 'soil' else (province_of(r.lat, r.lon) or site.province[r.soil_site])
                 for r in P.itertuples()]

# พืชในโมเดล -> (ชื่อพืช, คอลัมน์คำตอบจริง, เกณฑ์ไร่ขั้นต่ำ, ฤดูที่เทียบ)
ACTUAL = {'rice (นาปี)':   ('rice',    'rice_rai_2567',     100000, 'rainy'),
          'rice (นาปรัง)': ('rice',    'dry_rice_rai_2568', 100000, 'dry'),
          'maize':         ('maize',   'maize_rai_2567',     10000, 'rainy'),
          'coffee':        ('coffee',  'coffee_rai_2568',     1000, 'rainy'),
          'coconut':       ('coconut', 'coconut_rai_2568',    1000, 'rainy')}
truth = pd.read_csv('thailand_actual_crop_area.csv', index_col=0) if os.path.exists('thailand_actual_crop_area.csv') else None

if truth is None:
    print('ไม่พบ thailand_actual_crop_area.csv — ข้ามการตรวจ')
else:
    rows = []
    for label, (crop, col, thr, season) in ACTUAL.items():
        if col not in truth: continue
        q = predict(season); q['province'] = P.province.values
        hit = ((q.top1 == crop) | (q.top2 == crop) | (q.top3 == crop)).groupby(q.province).any()
        act = truth[col].reindex(hit.index).fillna(0) >= thr
        tp = int((hit & act).sum())
        rows.append({'พืช': label, 'ฤดูที่เทียบ': season, 'เกณฑ์ (ไร่)': thr, 'จังหวัดที่มีหมุด': len(hit),
                     'ปลูกจริง': int(act.sum()), 'โมเดลแนะนำ': int(hit.sum()), 'ตรงกัน': tp,
                     'แนะนำถูก %': round(100 * tp / max(hit.sum(), 1)), 'จับได้ %': round(100 * tp / max(act.sum(), 1))})
    eval_df = pd.DataFrame(rows)
    eval_df.to_csv(f'model_check{OUT}.csv', index=False, encoding='utf-8-sig')
    display(eval_df) if 'display' in globals() else print(eval_df.to_string(index=False))

          พืช ฤดูที่เทียบ  เกณฑ์ (ไร่)  จังหวัดที่มีหมุด  ปลูกจริง  โมเดลแนะนำ  ตรงกัน  แนะนำถูก %  จับได้ %
  rice (นาปี)       rainy       100000                67        47          43      28          65        60
rice (นาปรัง)         dry       100000                67        32           5       0           0         0
        maize       rainy        10000                67        27           0       0           0         0
       coffee       rainy         1000                67        19           3       3         100        16
      coconut       rainy         1000                67        23          46      16          35        70


## 8. แผนที่แบบมี dropdown เลือกพืชและฤดู
เมนูอยู่มุมขวาบนของแผนที่ เลือกแล้วหมุดเปลี่ยนทันที ไม่ต้องรัน cell ใหม่ · กดหมุดเพื่อดูว่า N, P, K, pH และอากาศของจุดนั้นเข้าช่วงที่พืชต้องการหรือไม่
ไฟล์ `thailand_crop_map_xgboost.html` ที่บันทึกก็มีเมนูนี้ด้วย เปิดในเบราว์เซอร์ได้โดยไม่ต้องมี Jupyter

In [30]:
from branca.element import Element

DROPDOWN_JS = """
window.addEventListener('load', function () {
  var map = __MAP__, D = __DATA__, layer = L.layerGroup().addTo(map);
  var SEA = {rainy: 'ฤดูฝน (พ.ค.–ต.ค.)', dry: 'ฤดูแล้ง (พ.ย.–เม.ย.)', year: 'ทั้งปี'};
  var FN = ['ไนโตรเจน N', 'ฟอสฟอรัส P', 'โพแทสเซียม K', 'อุณหภูมิ °C', 'ความชื้น %', 'pH', 'ฝน มม./เดือน'];
  var ctl = L.control({position: 'topright'});
  ctl.onAdd = function () {
    var d = L.DomUtil.create('div');
    d.style.cssText = 'background:#fff;color:#222;padding:8px 10px;border-radius:6px;box-shadow:0 1px 5px rgba(0,0,0,.4);font:14px sans-serif;line-height:1.9';
    d.innerHTML = 'พืช <select id="cropSel" style="font-size:14px"></select><br>ช่วง <select id="seaSel" style="font-size:14px"></select>' +
                  '<br><label id="actLbl"><input type="checkbox" id="actChk"> เฉพาะจังหวัดที่ปลูกจริง</label><div id="cropCnt" style="color:#555;font-size:12px"></div>';
    L.DomEvent.disableClickPropagation(d); L.DomEvent.disableScrollPropagation(d);
    return d;
  };
  ctl.addTo(map);
  var cs = document.getElementById('cropSel'), ss = document.getElementById('seaSel'), cnt = document.getElementById('cropCnt');
  var chk = document.getElementById('actChk'), lbl = document.getElementById('actLbl');
  Object.keys(SEA).forEach(function (k) { ss.add(new Option(SEA[k], k)); });
  function grown(p, crop, s) {            // null = ไม่มีสถิติของพืชนี้
    var a = D.act[crop]; if (!a) return null;
    var k = (crop === 'rice' && s === 'dry' && a.dry) ? 'dry' : 'main';
    return a[k][p.pv] === 1;
  }
  function count(c, s) { return D.pts.filter(function (p) { return p.s[s].t.some(function (t) { return t[0] === c; }); }).length; }
  function fill() {
    var cur = cs.value || '__CROP__';
    cs.innerHTML = ''; cs.add(new Option('ทุกพืช (อันดับ 1 ของแต่ละจุด)', 'all'));
    D.crops.slice().sort(function (a, b) { return count(b, ss.value) - count(a, ss.value); }).forEach(function (c) {
      cs.add(new Option(D.icon[c] + ' ' + D.th[c] + ' (' + count(c, ss.value) + ' จุด)', c)); });
    cs.value = cur;
  }
  function draw() {
    layer.clearLayers(); var n = 0, crop = cs.value, s = ss.value;
    D.pts.forEach(function (p) {
      var q = p.s[s], c, rank;
      if (crop === 'all') { c = q.t[0][0]; rank = 1; }
      else { rank = q.t.findIndex(function (t) { return t[0] === crop; }) + 1; if (!rank) return; c = crop; }
      var g = grown(p, c, s);
      if (chk.checked && g === false) return;
      n++;
      var html = '<div style="width:26px;height:26px;border-radius:50%;background:#fff;border:2px ' + (p.d > 100 ? 'dashed ' : 'solid ') +
                 (rank === 1 ? '#1f7a4d' : '#b7791f') + ';text-align:center;line-height:22px;font-size:14px">' + D.icon[c] + '</div>';
      var Rg = D.rng[c], ok = 0, rows = '';
      q.v.forEach(function (v, i) {
        var lo = Rg[i][0], hi = Rg[i][1], st = v < lo ? 'ต่ำไป' : v > hi ? 'สูงไป' : 'เข้าช่วง';
        if (st === 'เข้าช่วง') ok++;
        rows += '<tr><td>' + FN[i] + '</td><td style="text-align:right;padding:0 8px">' + v + '</td><td style="padding-right:8px">' + lo + '–' + hi +
                '</td><td style="font-weight:bold;color:' + (st === 'เข้าช่วง' ? '#1f7a4d' : '#b7791f') + '">' + st + '</td></tr>';
      });
      var pop = '<b>' + p.n + '</b><br>' + q.t.map(function (t, i) { return (i + 1) + '. ' + D.th[t[0]] + ' (' + t[1] + '%)'; }).join('<br>') +
                '<hr style="margin:4px 0"><b>' + D.th[c] + ': เข้าช่วง ' + ok + ' จาก 7 ปัจจัย</b>' +
                '<table style="font-size:12px;border-collapse:collapse;white-space:nowrap"><tr style="color:#666"><td>ปัจจัย</td><td style="text-align:right;padding:0 8px">จุดนี้</td><td>ช่วงที่พืชต้องการ</td><td></td></tr>' + rows + '</table>' +
                '<span style="color:#666">ช่วงที่พืชต้องการ = ค่าต่ำสุด–สูงสุดของพืชนั้นในชุดข้อมูล Kaggle</span>' +
                '<br>สถานีอากาศ: ' + p.st + ' · ระยะดิน–สถานี ' + p.d + ' กม.' +
                (g === null ? '' : '<br><b>จ.' + p.pv + (g ? ' ปลูก' : ' ไม่ได้ปลูก') + D.th[c] + 'จริงตามสถิติ สศก.</b>');
      L.marker([p.la, p.lo], {icon: L.divIcon({html: html, className: '', iconSize: [26, 26], iconAnchor: [13, 13]}), zIndexOffset: rank === 1 ? 500 : 0})
        .bindTooltip(p.n + ' – ' + D.th[c] + ' อันดับ ' + rank).bindPopup(pop, {maxWidth: 380}).addTo(layer);
    });
    var has = crop !== 'all' && !!D.act[crop];
    chk.disabled = !has; lbl.style.color = has ? '#222' : '#aaa';
    cnt.textContent = n + ' จุด · ขอบเขียว = อันดับ 1, ขอบส้ม = อันดับ 2–3' + (has ? '' : ' · พืชนี้ยังไม่มีสถิติปลูกจริง');
  }
  ss.value = '__SEASON__'; fill(); draw();
  cs.onchange = draw; chk.onchange = draw; ss.onchange = function () { fill(); draw(); };
});
"""

def crop_map_dropdown(crop='rice', season='rainy'):
    m = base_map()
    folium.LayerControl(position='bottomleft', collapsed=True).add_to(m)
    allres = {s_: predict(s_) for s_ in SEASONS}
    pts = []
    for i, r in enumerate(P.itertuples()):
        d = dict(n=r.name, pv=r.province, la=round(r.lat, 4), lo=round(r.lon, 4), st=r.station.title(), d=round(float(r.dist_km)), s={})
        for s_, df in allres.items():
            q = df.iloc[i]
            d['s'][s_] = dict(t=[[q.top1, round(q.prob1 * 100)], [q.top2, round(q.prob2 * 100)], [q.top3, round(q.prob3 * 100)]],
                              v=[float(q[f]) for f in FEATS])
        pts.append(d)
    act = {}
    if truth is not None:
        flag = lambda col, thr: {p: int(v >= thr) for p, v in truth[col].items()}
        for label, (c, col, thr, s_) in ACTUAL.items():
            if col in truth: act.setdefault(c, {})['dry' if (c == 'rice' and s_ == 'dry') else 'main'] = flag(col, thr)
    rng = {c: [[round(float(g[f].min()), 1), round(float(g[f].max()), 1)] for f in FEATS] for c, g in kag.groupby('label')}
    data = json.dumps(dict(pts=pts, icon=ICON, th=TH, crops=CROPS, act=act, rng=rng), ensure_ascii=False)
    js = (DROPDOWN_JS.replace('__MAP__', m.get_name()).replace('__DATA__', data)
                     .replace('__CROP__', crop).replace('__SEASON__', season))
    m.get_root().script.add_child(Element(js))
    return m

m2 = crop_map_dropdown('rice', SEASON)
m2.save(f'thailand_crop_map{OUT}.html')     # เปิดไฟล์นี้ในเบราว์เซอร์ได้ มี dropdown ในตัว
m2